# Simulation Studies with SimulationRunner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/03_simulation_studies.ipynb)

`SimulationRunner` is the research-oriented layer of `catsim`. It automates
repeated CAT sessions, keeps run-level exposure counts, and returns a
`SimulationResult` with aggregate metrics.

In [ ]:
from operator import itemgetter

import numpy as np

from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer
from catsim.item_bank import ItemBank
from catsim.selection import LinearSelector, MaxInfoSelector, RandomSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import TestLengthStopper

In [ ]:
item_bank = ItemBank.generate_item_bank(150, seed=31)
examinees = np.linspace(-2.0, 2.0, 40)

## A baseline simulation

In [ ]:
baseline = SimulationRunner(
  item_bank=item_bank,
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=TestLengthStopper(max_items=12),
  seed=99,
).run(examinees)

print("Bias:", round(baseline.bias, 3))
print("MSE:", round(baseline.mse, 3))
print("RMSE:", round(baseline.rmse, 3))
print("Overlap rate:", round(float(baseline.overlap_rate), 3))

## Integer examinee counts vs explicit theta arrays

Passing an integer asks `catsim` to generate a population from the item-bank
difficulty distribution. Passing an explicit array keeps the population fixed,
which is usually better for reproducible comparisons.

In [ ]:
generated_result = SimulationRunner(
  item_bank=item_bank,
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=TestLengthStopper(max_items=12),
  seed=99,
).run(40)

print("Generated population mean:", round(float(np.mean(generated_result.examinees)), 3))
print("Explicit population mean:", round(float(np.mean(examinees)), 3))

## Compare a few strategies

Here we compare one adaptive selector, one random selector, and one linear
baseline under the same item bank and explicit theta values.

In [ ]:
strategies = {
  "max_info": MaxInfoSelector(),
  "random": RandomSelector(),
  "linear": LinearSelector(list(range(12))),
}

rows = []
for name, selector in strategies.items():
  result = SimulationRunner(
    item_bank=item_bank,
    initializer=FixedPointInitializer(0.0),
    selector=selector,
    estimator=NumericalSearchEstimator(),
    stopper=TestLengthStopper(max_items=12),
    seed=99,
  ).run(examinees)
  rows.append({
    "selector": name,
    "bias": result.bias,
    "rmse": result.rmse,
    "overlap_rate": result.overlap_rate,
    "mean_length": np.mean([session.administered_count for session in result.sessions]),
  })

for row in sorted(rows, key=itemgetter("rmse")):
  printable = {key: round(float(value), 3) if key != "selector" else value for key, value in row.items()}
  print(printable)

## Interpretation notes

- `bias` shows systematic over- or under-estimation.
- `mse` and `rmse` summarize accuracy.
- `overlap_rate` is easier to interpret when all sessions have the same test length.
- `sessions` and `exposure_rates` let you move from aggregate metrics back to
  individual CAT behavior.

In methodological studies, it is usually better to compare strategies on the
same explicit examinee array rather than letting each run sample a different
synthetic population.